# ch09 交叉验证与调参（练习版）

> 按提示补全 `____`，跑通所有 assert。数据与管道 `pipe` 已在脚手架就绪。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。

In [ ]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, KFold,
                                     RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score, cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")

In [ ]:
# 数据与管道口径沿用 ch01/ch05：NUM 中位数填充+标准化，CAT 独热
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
X, y = dev[NUM + CAT], dev["是否危急"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
pipe = Pipeline([("pre", pre),
                 ("clf", LogisticRegression(max_iter=1000, random_state=42))])

print("sklearn", sklearn.__version__, "| 数据:", X.shape, "| 正例:", int(y.sum()))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def cv_summary(tag, scores):
    """一行输出 5 折均值/标准差。"""
    print(f"{tag}: mean={float(np.mean(scores)):.4f} std={float(np.std(scores)):.4f}")
    return float(np.mean(scores)), float(np.std(scores))


print("脚手架就绪：cv_summary")

## 任务 1：折结构对账

比较 KFold 与 StratifiedKFold 各 5 折的正例数分布。

In [ ]:
# TODO(1)：各折正例数对账：KFold 与 StratifiedKFold 各 5 折
# 提示：kf.split(X) 只吃 X；skf.split(X, y) 还要 y；折内 y.iloc[te].sum()
kf = ____
skf = ____
kf_pos = ____
skf_pos = ____
print("KFold 每折正例:", kf_pos)
print("StratifiedKFold 每折正例:", skf_pos, "| 总正例:", int(y.sum()))

assert kf_pos == [20, 21, 34, 27, 26], "普通 KFold 正例分布抖动 ±7"
assert skf_pos == [25, 25, 26, 26, 26], "分层后每折正例几乎一致"
assert sum(skf_pos) == 128

## 任务 2：单指标体检

分层 5 折跑 `cross_val_score`（roc_auc），输出各折与均值/标准差。

In [ ]:
# TODO(2)：用分层 5 折对管道跑 cross_val_score（roc_auc）
# 提示：cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
sc = ____
mean_auc, std_auc = cv_summary("cv AUC", sc)
print("各折:", np.round(sc, 4).tolist())

assert np.round(sc, 4).tolist() == [0.7676, 0.736, 0.7537, 0.7754, 0.84]
assert round(mean_auc, 4) == 0.7745 and round(std_auc, 4) == 0.0354

## 任务 3：多指标 + 训练分

`cross_validate` 同时要 roc_auc / f1 与 train 分数。

In [ ]:
# TODO(3)：多指标 + 训练分：cross_validate 同时要 roc_auc/f1 和 train 分数
# 提示：cross_validate(..., scoring=["roc_auc", "f1"], return_train_score=True)
cvr = ____
print("keys:", sorted(cvr.keys()))
print("test AUC:", np.round(cvr["test_roc_auc"], 4).tolist())
print("train AUC:", np.round(cvr["train_roc_auc"], 4).tolist(), "| test f1:",
      np.round(cvr["test_f1"], 4).tolist())

assert round(float(np.mean(cvr["test_roc_auc"])), 4) == 0.7745
assert round(float(np.mean(cvr["train_roc_auc"])), 4) == 0.8035, "train 略高于 test，正常"
assert round(float(np.mean(cvr["test_f1"])), 4) == 0.418, "f1 受阈值与不平衡双重影响"

## 任务 4：管道网格调参

定义 4×2 网格（`clf__C` × `clf__class_weight`）并跑 GridSearchCV。

In [ ]:
# TODO(4)：定义 4x2 网格（clf__C × clf__class_weight）并跑 GridSearchCV
# 提示：参数名是 管道步骤名__参数名；GridSearchCV(pipe, param_grid, cv=skf, scoring="roc_auc")
param_grid = ____
gs = ____
____
print("best params:", gs.best_params_)
print("best cv AUC:", round(float(gs.best_score_), 4), "| 组合数:", len(gs.cv_results_["params"]))

assert gs.best_params_ == {"clf__C": 10.0, "clf__class_weight": None}
assert round(float(gs.best_score_), 4) == 0.7748
assert len(gs.cv_results_["params"]) == 8, "4 x 2 = 8 组合，每组合 5 折 = 40 次拟合"

## 任务 5：解剖 cv_results_

提取 rank、best_index、最优组合的 train/test 分数与 gap。

In [ ]:
# TODO(5)：解剖 cv_results_：rank、best_index、最优组合的 train/test 差
# 提示：gs.best_index_；mean_test_score / mean_train_score 都是数组
best_idx = ____
rank = ____
best_test = ____
best_train = ____
print("rank:", rank)
print(f"best_index={best_idx} | train={best_train:.4f} vs test={best_test:.4f}")

assert rank == [8, 7, 4, 3, 2, 5, 1, 6]
assert best_idx == 6 and rank[best_idx] == 1
assert round(best_train, 4) == 0.8039 and round(best_test, 4) == 0.7748
assert round(best_train - best_test, 4) == 0.0291, "gap 不到 0.03：线性模型没过拟合"

## 任务 6：两阶段搜索

粗网格锁量级 → 细网格精修，验证最优 C 的一致性。

In [ ]:
# TODO(6)：两阶段搜索：粗网格锁量级，细网格精修（贝叶斯优化的穷替身）
# 提示：第一段粗网格已给；第二段把 C 网格压到 [3, 10, 30, 100]
gs_coarse = ____
____
coarse_best = ____
param_fine = ____
gs_fine = ____
____
print("粗网格 best C =", coarse_best, "→ 细网格:", param_fine["clf__C"])
print("细网格 best:", gs_fine.best_params_, round(float(gs_fine.best_score_), 4))

assert coarse_best == 10.0
assert abs(gs_fine.best_params_["clf__C"] - 10.0 / 3) < 1e-9, "细网格把最优推向 cb/3"
assert round(float(gs_fine.best_score_), 4) == 0.7749, "细网格比粗网格再挤 0.0001"

## 任务 7：随机搜索

`uniform` 连续分布抽 10 组，对比网格的成绩与成本。

In [ ]:
# TODO(7)：RandomizedSearchCV：连续分布上抽 10 组，看性价比
# 提示：uniform(0.01, 100) 表示 [0.01, 100.01) 均匀；RandomizedSearchCV(..., n_iter=10, random_state=42)
param_dist = ____
rs = ____
____
c_draws = [round(float(d["clf__C"]), 3) for d in rs.cv_results_["params"]]
print("抽样 C:", c_draws)
print("best:", {k: (round(float(v), 4) if isinstance(v, float) else v)
                for k, v in rs.best_params_.items()},
      round(float(rs.best_score_), 4))

assert c_draws == [37.464, 18.353, 59.876, 44.593, 5.818, 33.381, 70.817, 5.651, 83.254, 0.088]
assert rs.best_params_["clf__class_weight"] is None
assert round(float(rs.best_score_), 4) == 0.7749, "10 次随机 ≈ 8 组网格的成绩"

## 任务 8：树系网格与 refit 语义

RF 网格（f1 评分），确认 best_estimator_ 已在全量数据上重新拟合。

In [ ]:
# TODO(8)：树系网格 + refit 语义：best_estimator_ 已在全量数据上重新拟合
# 提示：Pipeline([("pre", pre), ("clf", RandomForestClassifier(random_state=42))])；网格 2x3
gs_rf = ____
____
rf_best_depth = ____
rf_best_score = ____
print("RF best:", gs_rf.best_params_, "| best f1:", round(rf_best_score, 4))
print("n_splits:", gs_rf.n_splits_, "| refit 耗时:", round(float(gs_rf.refit_time_), 2), "s")
print("best_estimator_ 可直接 predict:", gs_rf.best_estimator_.classes_.tolist())

assert gs_rf.best_params_ == {"clf__max_depth": None, "clf__n_estimators": 200}
assert round(rf_best_score, 4) == 0.296, "f1 在 21% 正例上依然很难看——选对 scoring 很重要"
assert gs_rf.n_splits_ == 5
assert gs_rf.best_estimator_.classes_.tolist() == [0, 1]

## 完成标准

所有 assert 通过 = 任务完成。自查：分层折为什么必要、`clf__C` 与 `C`
的区别、best_params_ 的统计意义、scoring 如何改变调参结论。